<a href="https://colab.research.google.com/github/kimmy111-zhu/human-validation/blob/main/notebooks/HellaSwag_Human_Validation_Sampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import json
import random
import pandas as pd
from google.colab import files

RANDOM_SEED = 42
SAMPLE_SIZE = 10
BENCHMARK_NAME = "HellaSwag"

FILE_PATHS = {
    "0.1": "/content/hellaswag_rate_0.1.jsonl",
    "0.4": "/content/hellaswag_rate_0.4.jsonl",
    "0.7": "/content/hellaswag_rate_0.7.jsonl"
}


def read_jsonl(file_path):
    records = []

    with open(file_path, "r", encoding="utf-8-sig") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            record = json.loads(line)
            record["_source_row"] = line_number
            records.append(record)

    return records


datasets = {}

for typo_rate, file_path in FILE_PATHS.items():
    datasets[typo_rate] = read_jsonl(file_path)
    print(
        f"Loaded {len(datasets[typo_rate])} records "
        f"for typo rate {typo_rate}"
    )


record_maps = {}

for typo_rate, records in datasets.items():
    record_maps[typo_rate] = {
        record["original_text_backup"]: record
        for record in records
        if record.get("original_text_backup")
    }


common_original_prompts = set(record_maps["0.1"].keys())

for typo_rate in ["0.4", "0.7"]:
    common_original_prompts &= set(record_maps[typo_rate].keys())

common_original_prompts = sorted(common_original_prompts)

print(f"Common original prompts: {len(common_original_prompts)}")


if len(common_original_prompts) < SAMPLE_SIZE:
    raise ValueError(
        f"Only {len(common_original_prompts)} matched prompts were found. "
        f"Cannot sample {SAMPLE_SIZE} prompts."
    )


random.seed(RANDOM_SEED)

selected_original_prompts = random.sample(
    common_original_prompts,
    SAMPLE_SIZE
)


output_rows = []

for prompt_number, original_prompt in enumerate(
    selected_original_prompts,
    start=1
):
    base_id = f"{BENCHMARK_NAME}_{prompt_number:03d}"

    for typo_rate in ["0.1", "0.4", "0.7"]:
        record = record_maps[typo_rate][original_prompt]

        output_rows.append({
            "Base_ID": base_id,
            "Sample_ID": f"{base_id}_rate_{typo_rate}",
            "Benchmark": BENCHMARK_NAME,
            "Typo_Rate": typo_rate,
            "Source_Row": record.get("_source_row", ""),
            "Original_Prompt": original_prompt,
            "Modified_Prompt": record.get("question", ""),
            "Gold_Answer": record.get("answer", ""),
            "R1_Meaning": "",
            "R2_Meaning": "",
            "Final_Meaning": "",
            "R1_Key_Info": "",
            "R2_Key_Info": "",
            "Final_Key_Info": "",
            "R1_Realism": "",
            "R2_Realism": "",
            "Final_Realism": "",
            "R1_Readability": "",
            "R2_Readability": "",
            "Final_Readability": "",
            "Comments": ""
        })


sample_df = pd.DataFrame(output_rows)

print(f"Random seed: {RANDOM_SEED}")
print(f"Selected original prompts: {SAMPLE_SIZE}")
print(f"Total output rows: {len(sample_df)}")

display(sample_df)


output_file = "/content/HellaSwag_matched_sample_n10_seed42.csv"

sample_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"CSV saved successfully: {output_file}")

files.download(output_file)

Loaded 39905 records for typo rate 0.1
Loaded 39905 records for typo rate 0.4
Loaded 39905 records for typo rate 0.7
Common original prompts: 39799
Random seed: 42
Selected original prompts: 10
Total output rows: 30


,Base_ID,Sample_ID,Benchmark,Typo_Rate,Source_Row,Original_Prompt,Modified_Prompt,Gold_Answer,R1_Meaning,R2_Meaning,...,R1_Key_Info,R2_Key_Info,Final_Key_Info,R1_Realism,R2_Realism,Final_Realism,R1_Readability,R2_Readability,Final_Readability,Comments
0,HellaSwag_001,HellaSwag_001_rate_0.1,HellaSwag,0.1,5977,A young woman is seen kneeling in a tub with c...,,,,,...,,,,,,,,,,
1,HellaSwag_001,HellaSwag_001_rate_0.4,HellaSwag,0.4,5977,A young woman is seen kneeling in a tub with c...,,,,,...,,,,,,,,,,
2,HellaSwag_001,HellaSwag_001_rate_0.7,HellaSwag,0.7,5977,A young woman is seen kneeling in a tub with c...,,,,,...,,,,,,,,,,
3,HellaSwag_002,HellaSwag_002_rate_0.1,HellaSwag,0.1,4597,A group of people are on a basketball court. T...,,,,,...,,,,,,,,,,
4,HellaSwag_002,HellaSwag_002_rate_0.4,HellaSwag,0.4,4597,A group of people are on a basketball court. T...,,,,,...,,,,,,,,,,
5,HellaSwag_002,HellaSwag_002_rate_0.7,HellaSwag,0.7,4597,A group of people are on a basketball court. T...,,,,,...,,,,,,,,,,
6,HellaSwag_003,HellaSwag_003_rate_0.1,HellaSwag,0.1,31559,[header] How to care for newborn hamsters [tit...,,,,,...,,,,,,,,,,
7,HellaSwag_003,HellaSwag_003_rate_0.4,HellaSwag,0.4,31559,[header] How to care for newborn hamsters [tit...,,,,,...,,,,,,,,,,
8,HellaSwag_003,HellaSwag_003_rate_0.7,HellaSwag,0.7,31559,[header] How to care for newborn hamsters [tit...,,,,,...,,,,,,,,,,
9,HellaSwag_004,HellaSwag_004_rate_0.1,HellaSwag,0.1,37505,[header] How to be a pious young muslimah [tit...,,,,,...,,,,,,,,,,


CSV saved successfully: /content/HellaSwag_matched_sample_n10_seed42.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>